In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
DATASET_PATH = "../data/train.csv"

LOW_QUANTILE = 0.10
HIGH_QUANTILE = 0.90
IRQ_COEFF = 3

# Categorical encoding

In [3]:
ds = pd.read_csv(DATASET_PATH)

important_df =  ds[['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'Fee', 'State', 'AdoptionSpeed']]

important_df = important_df.drop(columns=['Color3', 'Fee', 'State'])

In [4]:
important_df.head(5)

,Age,Breed1,Breed2,Gender,Color1,Color2,MaturitySize,Vaccinated,FurLength,Dewormed,Sterilized,Health,Quantity,AdoptionSpeed
0,3,299,0,1,1,7,1,2,1,2,2,1,1,2
1,1,265,0,1,1,2,2,3,2,3,3,1,1,0
2,1,307,0,1,2,7,2,1,2,1,2,1,1,3
3,4,307,0,2,1,2,2,1,1,1,2,1,1,2
4,1,307,0,1,1,0,2,2,1,2,2,1,1,2


# Handling Nan values

In [5]:
print("Any missing sample in training set:",ds.isnull().values.any())

Any missing sample in training set: True


In [6]:
for col in ds.columns:
    if ds[col].isnull().values.any():
        print("Missing data in ", col)

Missing data in  Name
Missing data in  Description


In [7]:
missing = list()
for x in ds.columns:
    if ds[x].isnull().sum() != 0:
        print(f"{x:<30}{ds[x].isnull().sum():<10}{(ds[x].isnull().sum() / ds.shape[0])*100}%")
        missing.append(x)

Name                          1265      8.437270726338959%
Description                   13        0.0867071299939972%


# Handling Anomalies

In [8]:
def detect_anomalies(df, column):
    Q1 = df[column].quantile(LOW_QUANTILE)
    Q3 = df[column].quantile(HIGH_QUANTILE)
    IQR = Q3 - Q1
    lower_bound = Q1 - IRQ_COEFF * IQR
    upper_bound = Q3 + IRQ_COEFF * IQR
    anomalies = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return anomalies

numerical_columns = ['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'AdoptionSpeed']

for col in numerical_columns:
    anomalies = detect_anomalies(important_df, col)
    print(f"Anomalies in {col}:")
    print(anomalies.shape[0])

Anomalies in Age:
62
Anomalies in Breed1:
0
Anomalies in Breed2:
0
Anomalies in Gender:
0
Anomalies in Color1:
0
Anomalies in Color2:
0
Anomalies in MaturitySize:
0
Anomalies in Vaccinated:
0
Anomalies in FurLength:
0
Anomalies in Dewormed:
0
Anomalies in Sterilized:
0
Anomalies in Health:
515
Anomalies in Quantity:
62
Anomalies in AdoptionSpeed:
0


In [9]:
def replace_anomalies_with_minmax_values(df, column):
    sorted_values = df[column].sort_values()
    Q1 = sorted_values.quantile(LOW_QUANTILE)
    Q3 = sorted_values.quantile(HIGH_QUANTILE)
    IQR = Q3 - Q1
    lower_bound = Q1 - IRQ_COEFF * IQR
    upper_bound = Q3 + IRQ_COEFF * IQR

    min_real_value = sorted_values[sorted_values >= lower_bound].min()
    max_real_value = sorted_values[sorted_values <= upper_bound].max()

    df.loc[df[column] < lower_bound, column] = min_real_value
    df.loc[df[column] > upper_bound, column] = max_real_value

numerical_columns = ['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'AdoptionSpeed']

for col in numerical_columns:
    replace_anomalies_with_minmax_values(important_df, col)

print("Anomalies replaced with real values. Updated DataFrame:")
important_df.head(5)

Anomalies replaced with real values. Updated DataFrame:


,Age,Breed1,Breed2,Gender,Color1,Color2,MaturitySize,Vaccinated,FurLength,Dewormed,Sterilized,Health,Quantity,AdoptionSpeed
0,3,299,0,1,1,7,1,2,1,2,2,1,1,2
1,1,265,0,1,1,2,2,3,2,3,3,1,1,0
2,1,307,0,1,2,7,2,1,2,1,2,1,1,3
3,4,307,0,2,1,2,2,1,1,1,2,1,1,2
4,1,307,0,1,1,0,2,2,1,2,2,1,1,2


# Scaling

In [10]:
def scale_columns(df, columns):
    for col in columns:
        mean = df[col].mean()
        std = df[col].std()
        df[col] = (df[col] - mean) / std

numerical_columns = ['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'AdoptionSpeed']
scale_columns(important_df, numerical_columns)

print("Numerical columns scaled using z-score normalization:")
important_df.head(5)

Numerical columns scaled using z-score normalization:


,Age,Breed1,Breed2,Gender,Color1,Color2,MaturitySize,Vaccinated,FurLength,Dewormed,Sterilized,Health,Quantity,AdoptionSpeed
0,-0.425064,0.561592,-0.601649,-1.138749,-0.707173,1.377239,-1.573115,0.402595,-0.780351,0.634179,0.151497,NaN,-0.422538,-0.438679
1,-0.541046,-0.004539,-0.601649,-1.138749,-0.707173,-0.445874,0.251840,1.900387,0.888903,2.071339,1.917745,NaN,-0.422538,-2.137532
2,-0.541046,0.694799,-0.601649,-1.138749,-0.134181,1.377239,0.251840,-1.095198,0.888903,-0.802981,0.151497,NaN,-0.422538,0.410748
3,-0.367074,0.694799,-0.601649,0.328404,-0.707173,-0.445874,0.251840,-1.095198,-0.780351,-0.802981,0.151497,NaN,-0.422538,-0.438679
4,-0.541046,0.694799,-0.601649,-1.138749,-0.707173,-1.175119,0.251840,0.402595,-0.780351,0.634179,0.151497,NaN,-0.422538,-0.438679
